# Custom Oracles in LearnM8 (Run Mode)

Learn how to use example oracles for real-world molecular screening.

**What you'll learn:**
- Use SimilarityOracle with greedy selection
- Use Pharmacophore2DOracle with UCB (custom parameters)
- Use VinaOracle with greedy + pruning
- Understand run mode vs benchmark mode metrics

**Key Difference from Notebook 1:**
- **Run Mode**: Oracle generates values on-the-fly (no ground truth)
- **Metrics**: Average selection metrics (not enrichment/discovery)
- **Realistic**: Simulates prospective screening campaigns

**Time estimate:** 15 minutes

## Section 1: SimilarityOracle with Greedy Selection

In [ ]:
from examples.oracles import SimilarityOracle
from learnm8 import run_active_learning
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt

# Load compound library (1K compounds, ID + SMILES only, NO ground truth)
compounds = pl.read_csv('../data/ampc_1k_no_scores.csv')
print(f"Loaded {len(compounds)} compounds")
print(f"Columns: {compounds.columns}")
print("Note: No ground truth available - oracle will generate similarity scores on-the-fly")

# Create similarity oracle targeting phenol-like structures
oracle = SimilarityOracle(
    reference_smiles='c1ccccc1O',  # Phenol scaffold
    fingerprint_type='morgan',
    metric='tanimoto',
    radius=2,
    n_bits=2048
)

# Run active learning with GREEDY strategy (RUN MODE)
results = run_active_learning(
    compound_pool=compounds,         # DataFrame with ID and SMILES only
    oracle=oracle,                   # SimilarityOracle computes scores on-demand
    target_col='similarity',         # Property name oracle will populate
    learner='rf',                    # Random Forest (greedy doesn't need uncertainty)
    featurizer_type='morgan',        # Morgan fingerprints
    strategy='greedy',               # Select compounds with highest predicted similarity
    n_cycles=5,                      # 5 active learning cycles
    batch_fraction=0.02,             # Select 2% of pool per cycle
    score_direction='higher'         # Higher similarity is better
)

print(f"\n✓ Experiment complete!")
print(f"  Compounds measured: {len(results['measured_ids'])}")

# RUN MODE METRICS: Report average similarity of selected compounds (not enrichment)
measured_df = results['compounds_df'].filter(pl.col('measured'))
print(f"  Average similarity (selected): {measured_df['similarity'].mean():.3f}")
print(f"  Top similarity found: {measured_df['similarity'].max():.3f}")

# Plot selection metrics over cycles
metrics_df = pd.DataFrame(results['cycle_metrics'])

# Calculate best score found so far per cycle
measured_df = results['compounds_df'].filter(pl.col('measured'))
best_scores_per_cycle = []
for cycle_num in range(len(metrics_df)):
    cycle_measured = measured_df.filter(pl.col('cycle_measured') <= cycle_num)
    best_scores_per_cycle.append(cycle_measured['similarity'].max())

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(metrics_df['cycle'], metrics_df['avg_score_selected'], marker='o', linewidth=2)
plt.xlabel('Cycle')
plt.ylabel('Avg Similarity (Selected)')
plt.title('Average Similarity of Selected Compounds')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(metrics_df['cycle'], best_scores_per_cycle, marker='s', linewidth=2, color='green')
plt.xlabel('Cycle')
plt.ylabel('Best Similarity Found')
plt.title('Best Score Discovery Progress')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.plot(metrics_df['cycle'], metrics_df['n_measured'].cumsum(), marker='^', linewidth=2, color='orange')
plt.xlabel('Cycle')
plt.ylabel('Cumulative Measurements')
plt.title('Measurement Budget Usage')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 2: Pharmacophore2DOracle with UCB (Custom Parameters)

In [ ]:
from examples.oracles import Pharmacophore2DOracle

# Create pharmacophore oracle targeting salicylic acid-like features
ph4_oracle = Pharmacophore2DOracle(
    reference_smiles='c1ccc(O)cc1C(=O)O',  # Salicylic acid
    metric='tanimoto'
)

# Run with UCB strategy (requires uncertainty quantification)
# UCB balances exploitation (high prediction) and exploration (high uncertainty)
ph4_results = run_active_learning(
    compound_pool=compounds,                # Same 1K compound pool
    oracle=ph4_oracle,                      # Pharmacophore oracle
    target_col='pharmacophore_similarity',  # Property name
    learner='rf_ensemble',                  # RF Ensemble provides uncertainty estimates
    featurizer_type='morgan',               # Morgan fingerprints
    strategy='ucb',                         # Upper Confidence Bound strategy
    acquisition_params={'beta': 2.0},       # Custom UCB parameter: exploration weight
                                            # beta=2.0 (default), higher=more exploration
    n_cycles=5,                             # 5 cycles
    batch_fraction=0.02,                    # 2% per cycle
    score_direction='higher'                # Higher pharmacophore similarity is better
)

print(f"\n✓ Pharmacophore + UCB screening complete")
print(f"  Compounds measured: {len(ph4_results['measured_ids'])}")

# RUN MODE METRICS: Report selection quality (not enrichment)
measured_df = ph4_results['compounds_df'].filter(pl.col('measured'))
print(f"  Average pharmacophore similarity (selected): {measured_df['pharmacophore_similarity'].mean():.3f}")
print(f"  Top similarity found: {measured_df['pharmacophore_similarity'].max():.3f}")

# Plot UCB-specific metrics
ph4_metrics_df = pd.DataFrame(ph4_results['cycle_metrics'])

# Calculate best pharmacophore score found so far per cycle
ph4_measured_df = ph4_results['compounds_df'].filter(pl.col('measured'))
ph4_best_scores = []
for cycle_num in range(len(ph4_metrics_df)):
    cycle_measured = ph4_measured_df.filter(pl.col('cycle_measured') <= cycle_num)
    ph4_best_scores.append(cycle_measured['pharmacophore_similarity'].max())

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(ph4_metrics_df['cycle'], ph4_metrics_df['avg_score_selected'],
         marker='o', linewidth=2, label='Avg Score')
plt.xlabel('Cycle')
plt.ylabel('Avg Pharmacophore Similarity')
plt.title('UCB Selection Quality')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(ph4_metrics_df['cycle'], ph4_best_scores,
         marker='s', linewidth=2, color='green', label='Best Score')
plt.xlabel('Cycle')
plt.ylabel('Best Pharmacophore Similarity')
plt.title('Best Score Discovery Progress')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.plot(ph4_metrics_df['cycle'], ph4_metrics_df['avg_uncertainty_selected'],
         marker='^', linewidth=2, color='orange', label='Avg Uncertainty')
plt.xlabel('Cycle')
plt.ylabel('Average Uncertainty')
plt.title('UCB Exploration Behavior')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 UCB trades off exploitation (high predictions) vs exploration (high uncertainty)")
print(f"   beta=2.0 used - increase for more exploration, decrease for more exploitation")

## Section 3: VinaOracle with Greedy + Pruning

In [ ]:
from examples.oracles import VinaOracle
from pathlib import Path

# Note: VinaOracle requires receptor structure and configuration
# For this example, ensure you have a receptor PDB file and config

print("\n=== Experiment 3: VinaOracle with Greedy + Pruning ===")
print("⚠️  This experiment requires:")
print("    - Receptor PDB file")
print("    - Vina configuration (search space, exhaustiveness)")
print("    - ~10-60s per docking calculation")

# Example receptor path (adjust to your files)
receptor_pdb = Path('receptor.pdbqt')  # Prepared receptor file

if receptor_pdb.exists():
    # Create Vina oracle for molecular docking
    vina_oracle = VinaOracle(
        receptor_pdbqt=str(receptor_pdb),
        center=(10.0, 5.0, 15.0),     # Binding site center (x, y, z)
        box_size=(20.0, 20.0, 20.0),  # Search box dimensions
        exhaustiveness=8               # Docking thoroughness (default=8)
    )

    # Run with PRUNING to reduce expensive docking calls
    # Pruning: Remove unlikely candidates after initial rounds
    vina_results = run_active_learning(
        compound_pool=compounds.head(100),  # Use small subset (docking is expensive!)
        oracle=vina_oracle,                 # Vina oracle
        target_col='docking_score',         # Property name
        learner='rf',                       # Random Forest
        featurizer_type='morgan',           # Morgan fingerprints
        strategy='greedy',                  # Greedy selection
        n_cycles=3,                         # Limited cycles (expensive oracle)
        batch_fraction=0.05,                # 5% per cycle
        score_direction='lower',            # Lower docking score is better
        pruning_strategy='score_based',     # Prune low-scoring compounds
        pruning_params={
            'pruning_fraction': 0.5,        # Remove bottom 50% after cycle 1
            'start_cycle': 1                # Start pruning after cycle 1
        }
    )

    print(f"\n✓ Vina + Pruning screening complete")
    print(f"  Compounds docked: {len(vina_results['measured_ids'])}")

    # RUN MODE METRICS: Report docking results
    docked_df = vina_results['compounds_df'].filter(pl.col('measured'))
    print(f"  Average docking score: {docked_df['docking_score'].mean():.2f} kcal/mol")
    print(f"  Best docking score: {docked_df['docking_score'].min():.2f} kcal/mol")
    print(f"  Pruned pool size: {len(vina_results['compounds_df'].filter(~pl.col('pruned')))} compounds")

    # Plot pruning impact
    vina_metrics_df = pd.DataFrame(vina_results['cycle_metrics'])

    # Calculate best docking score found so far per cycle
    vina_measured_df = vina_results['compounds_df'].filter(pl.col('measured'))
    vina_best_scores = []
    for cycle_num in range(len(vina_metrics_df)):
        cycle_measured = vina_measured_df.filter(pl.col('cycle_measured') <= cycle_num)
        vina_best_scores.append(cycle_measured['docking_score'].min())  # min for lower is better

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.plot(vina_metrics_df['cycle'], vina_metrics_df['avg_score_selected'],
             marker='o', linewidth=2)
    plt.xlabel('Cycle')
    plt.ylabel('Avg Docking Score (kcal/mol)')
    plt.title('Greedy Selection Quality')
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 3, 2)
    plt.plot(vina_metrics_df['cycle'], vina_best_scores,
             marker='s', linewidth=2, color='green')
    plt.xlabel('Cycle')
    plt.ylabel('Best Docking Score (kcal/mol)')
    plt.title('Best Score Discovery Progress')
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 3, 3)
    plt.plot(vina_metrics_df['cycle'], vina_metrics_df['pool_size_remaining'],
             marker='^', linewidth=2, color='red')
    plt.xlabel('Cycle')
    plt.ylabel('Pool Size Remaining')
    plt.title('Pruning Effect')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"\n💡 Pruning removes unlikely candidates to reduce expensive oracle calls")
    print(f"   50% of pool pruned after cycle 1 based on predicted scores")

else:
    print(f"\n⚠️  Vina oracle requires receptor file: {receptor_pdb}")
    print("   Skipping VinaOracle example (optional dependency)")
    print("   Install Vina: conda install -c conda-forge vina meeko")

## Key Takeaways

**Run Mode vs Benchmark Mode:**
- **Benchmark Mode** (Notebook 1): Ground truth known → enrichment/discovery metrics
- **Run Mode** (Notebook 2): Oracle generates values on-the-fly → selection metrics only

**Three Oracle Examples:**
1. **SimilarityOracle + Greedy**: Fast 2D similarity, exploitation-focused
2. **Pharmacophore2DOracle + UCB**: Feature matching with exploration/exploitation balance
3. **VinaOracle + Pruning**: Expensive docking with design space reduction

**Metrics in Run Mode:**
- ✅ Average score of selected compounds
- ✅ Top score found
- ✅ Cumulative measurements
- ✅ Uncertainty trends (for UCB/ensemble)
- ❌ Enrichment factor (requires ground truth)
- ❌ Top-K discovery (requires ground truth)

**Key Learnings:**
- `learner='rf'` for greedy strategy (no uncertainty needed)
- `learner='rf_ensemble'` for UCB (provides uncertainty)
- `acquisition_params` customize strategy behavior (e.g., beta for UCB)
- `pruning_strategy` reduces oracle calls for expensive measurements

**Next:** [03_advanced_configuration.ipynb](03_advanced_configuration.ipynb) for advanced configuration